# Step 7: compare tuned and untuned rankers

Use the same complete training data, fixed trip folds, feature list and metric rules for both configurations. The untuned model uses XGBoost's installed defaults, with only objective, metric, histogram algorithm, seed and CPU count specified. It is not the previously tuned Step 6 model.

The tuned model uses the configuration frozen by the 50-trial search. Its full-row CV score is still subject to hyperparameter-selection bias. Student-test results are computed only after that configuration is frozen, and are not used to change it.

This notebook regenerates tuned OOF probabilities and fits final models. Stacking and HMM results require their separate model artifacts; missing artifacts are reported explicitly.

In [1]:
%run 06_model_helpers.ipynb
import xgboost as xgb
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo
import matplotlib.pyplot as plt

search_output = Path("outputs/step7_xgb_sample10000")
output_root = Path("results/step7_xgb")
output_root.mkdir(parents=True, exist_ok=True)
n_jobs = 8
data, points = load_model_data()
metric_names = ["top1", "macro_f1", "precision", "recall", "log_loss", "candidate_coverage"]

In [2]:
def fit_ranker(frame, params):
    sizes = frame.groupby("group_id")["label"].agg(["size", "sum"])
    eligible = sizes.index[(sizes["size"] >= 2) & (sizes["sum"] == 1)]
    training = frame[frame["group_id"].isin(eligible)]
    assert training["group_id"].is_monotonic_increasing
    groups = training.groupby("group_id", sort=False).size().to_numpy()
    model = xgb.XGBRanker(objective="rank:ndcg", eval_metric="ndcg@1", tree_method="hist",
                          random_state=seed, n_jobs=n_jobs, **params)
    model.fit(training[feature_columns], training["label"], group=groups, verbose=False)
    return model


## Full-row CV, OOF and final refits

Train on all eligible groups in the other four folds. Predict all valid held-out candidate rows, including singleton and no-positive groups. Save OOF scores as float64 so pooling does not introduce extra probability-rounding ties.

Each completed variant saves its CV metrics, OOF table and final model. Reuse a finished variant when rerunning the comparison; unfinished variants are rerun. The saved params, row count, predictor list and fold scheme must match.

In [3]:
def run_variant(name, params):
    folder = output_root / name
    folder.mkdir(parents=True, exist_ok=True)
    signature = {"params": params, "seed": seed, "rows": len(data), "features": feature_columns,
                 "fold_scheme": contract["fold_scheme"], "candidate_settings": contract["candidate_settings"]}
    signature_file = folder / "run_config.json"
    if signature_file.exists():
        assert json.loads(signature_file.read_text()) == signature
    signature_file.write_text(json.dumps(signature, indent=2) + "\n")
    required = ["cv_metrics.csv", "pooled_metrics.json", "timings.json", "model.json", f"oof_{name}.csv"]
    if all((folder / path).exists() for path in required):
        print("Reusing completed variant:", name, flush=True)
        return folder

    probability_oof = np.full(len(data), np.nan, dtype=np.float64)
    raw_oof = np.full(len(data), np.nan, dtype=np.float32)
    records = []
    for fold in range(5):
        training = data[data["fold"] != fold]
        validation = data[data["fold"] == fold]
        valid_points = points[points["fold"] == fold]
        assert set(training["trip_id"]).isdisjoint(validation["trip_id"])
        start = perf_counter()
        model = fit_ranker(training, params)
        fit_s = perf_counter() - start
        start = perf_counter()
        raw = model.predict(validation[feature_columns])
        probability = point_softmax(raw, validation["group_id"])
        predict_s = perf_counter() - start
        probability_oof[validation.index] = probability
        raw_oof[validation.index] = raw
        records.append({"fold": fold, **score_points(validation, probability, valid_points),
                        "fit_s": fit_s, "predict_s": predict_s})
        pd.DataFrame(records).to_csv(folder / "progress.csv", index=False)
        print(name, "fold", fold, "Top-1", records[-1]["top1"], flush=True)

    assert np.isfinite(probability_oof).all() and np.isfinite(raw_oof).all()
    oof = data[["group_id", "segment_id", "fold"]].copy()
    oof["score"], oof["raw_score"] = probability_oof, raw_oof
    oof.to_csv(folder / f"oof_{name}.csv", index=False)
    cv = pd.DataFrame(records)
    cv.to_csv(folder / "cv_metrics.csv", index=False)
    pooled = score_points(data, probability_oof, points)
    assert np.array_equal(np.sum([r["confusion_matrix"] for r in records], axis=0), pooled["confusion_matrix"])
    (folder / "pooled_metrics.json").write_text(json.dumps(pooled, indent=2) + "\n")
    start = perf_counter()
    model = fit_ranker(data, params)
    final_fit_s = perf_counter() - start
    model.save_model(folder / "model.json")
    (folder / "booster_config.json").write_text(model.get_booster().save_config())
    timings = {"final_fit_s": final_fit_s, "fit_s_mean": cv["fit_s"].mean(),
               "predict_s_mean": cv["predict_s"].mean(), "xgboost_version": xgb.__version__}
    (folder / "timings.json").write_text(json.dumps(timings, indent=2) + "\n")
    return folder

In [4]:
baseline_folder = run_variant("untuned", {})
print("Untuned full-row CV and final refit complete. Student-test scores remain unopened.")

untuned fold 0 Top-1 0.5124481842778067


untuned fold 1 Top-1 0.8554205316267587


untuned fold 2 Top-1 0.921101181206124


untuned fold 3 Top-1 0.8307733178766362


untuned fold 4 Top-1 0.895558631407688


Untuned full-row CV and final refit complete. Student-test scores remain unopened.
